# Stage 1: Data preprocessing [html](https://scglue.readthedocs.io/en/latest/preprocessing.html#Stage-1:-Data-preprocessing)

In [1]:
import anndata as ad
import networkx as nx
import scanpy as sc
import scglue
from matplotlib import rcParams

/opt/conda/lib/python3.8/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [3]:
scglue.plot.set_publication_params()
rcParams["figure.figsize"] = (4, 4)

## Read data

- http://download.gao-lab.org/GLUE/tutorial/Chen-2019-RNA.h5ad
- http://download.gao-lab.org/GLUE/tutorial/Chen-2019-ATAC.h5ad

In [4]:
rna = ad.read_h5ad("./demo/Chen-2019-RNA.h5ad")
rna

AnnData object with n_obs × n_vars = 9190 × 28930
    obs: 'domain', 'cell_type'

In [5]:
atac = ad.read_h5ad("./demo/Chen-2019-ATAC.h5ad")
atac

AnnData object with n_obs × n_vars = 9190 × 241757
    obs: 'domain', 'cell_type'

## Preprocess scRNA-seq data

- `.layers['counts']` is raw data
- `.X` is normalized data
- `cell_type` is annotation key

In [6]:
rna.X, rna.X.data

(<9190x28930 sparse matrix of type '<class 'numpy.float32'>'
 	with 8633857 stored elements in Compressed Sparse Row format>,
 array([1., 1., 1., ..., 1., 1., 1.], dtype=float32))

In [7]:
rna.layers["counts"] = rna.X.copy()

In [8]:
sc.pp.highly_variable_genes(rna, n_top_genes=2000, flavor="seurat_v3")

ImportError: Please install skmisc package via `pip install --user scikit-misc

In [ ]:
sc.pp.normalize_total(rna)
sc.pp.log1p(rna)
sc.pp.scale(rna)
sc.tl.pca(rna, n_comps=100, svd_solver="auto")

In [ ]:
sc.pp.neighbors(rna, metric="cosine")
sc.tl.umap(rna)
sc.pl.umap(rna, color="cell_type")

## Preprocess scATAC-seq data

In [ ]:
# scATAC-seq accessibility matrix is also supposed to contain raw counts
atac.X, atac.X.data

In [ ]:
scglue.data.lsi(atac, n_components=100, n_iter=15)

In [ ]:
sc.pp.neighbors(atac, use_rep="X_lsi", metric="cosine")
sc.tl.umap(atac)

In [ ]:
sc.pl.umap(atac, color="cell_type")

## Construct prior regulatory graph

- The graph should contain omics features as nodes (e.g., genes for scRNA-seq, and peaks for scATAC-seq), and prior regulatory interactions as edges.
- use **GLUE construct graph** or **custom specifical graph**

**obstain genomic coordinates**

In [23]:
rna.var.head()

,highly_variable,highly_variable_rank,means,variances,variances_norm,mean,std
genes,,,,,,,
0610005C13Rik,False,NaN,0.001415,0.001413,0.958918,0.000832,0.024802
0610009B22Rik,True,1528.0,0.017301,0.022228,1.126013,0.010283,0.091634
0610009E02Rik,False,NaN,0.014799,0.017193,1.023411,0.008182,0.076223
0610009L18Rik,False,NaN,0.016540,0.020403,1.082735,0.009849,0.088678
0610010F05Rik,False,NaN,0.157563,0.197176,1.004244,0.086493,0.245927


In [24]:
scglue.data.get_gene_annotation(
    rna, gtf="./demo/gencode.vM25.chr_patch_hapl_scaff.annotation.gtf.gz",
    gtf_by="gene_name"
)
rna.var.loc[:, ["chrom", "chromStart", "chromEnd"]].head()

,chrom,chromStart,chromEnd
genes,,,
0610005C13Rik,chr7,45567793,45575327
0610009B22Rik,chr11,51685385,51688874
0610009E02Rik,chr2,26445695,26459390
0610009L18Rik,chr11,120348677,120351190
0610010F05Rik,chr11,23564960,23633639


In [25]:
atac.var_names[:5]

Index(['chr1:3005833-3005982', 'chr1:3094772-3095489', 'chr1:3119556-3120739',
       'chr1:3121334-3121696', 'chr1:3134637-3135032'],
      dtype='object', name='peaks')

In [26]:
split = atac.var_names.str.split(r"[:-]")
atac.var["chrom"] = split.map(lambda x: x[0])
atac.var["chromStart"] = split.map(lambda x: x[1]).astype(int)
atac.var["chromEnd"] = split.map(lambda x: x[2]).astype(int)
atac.var.head()

,chrom,chromStart,chromEnd
peaks,,,
chr1:3005833-3005982,chr1,3005833,3005982
chr1:3094772-3095489,chr1,3094772,3095489
chr1:3119556-3120739,chr1,3119556,3120739
chr1:3121334-3121696,chr1,3121334,3121696
chr1:3134637-3135032,chr1,3134637,3135032


**Graph construction**

In [28]:
import scglue

In [31]:
help(scglue.genomics.rna_anchored_guidance_graph)

AttributeError: module 'scglue.genomics' has no attribute 'rna_anchored_guidance_graph'

In [29]:
guidance = scglue.genomics.rna_anchored_guidance_graph(rna, atac)
guidance

AttributeError: module 'scglue.genomics' has no attribute 'rna_anchored_guidance_graph'

In [ ]:
scglue.graph.check_graph(guidance, [rna, atac])

In [ ]:
atac.var.head()

**save reprocessed data files**

In [ ]:
rna.write("rna-pp.h5ad", compression="gzip")
atac.write("atac-pp.h5ad", compression="gzip")
nx.write_graphml(guidance, "guidance.graphml.gz")